#Nesta versão, tiramos a tarnsparência

*   Item da lista
*   Item da lista



## Roteiro de geração (pré-configurado)

São 6 execuções. Em cada uma, mude só `ESPECIME` e `ETAPA` no **Painel** e rode, em ordem: Painel → célula do v4 → **Renomear saída**.

| # | `ESPECIME` | `ETAPA` | Gera |
|---|---|---|---|
| 1 | `"mariposa"` | `"canonico"` | 1 `.ply` (modelo canônico após o pruning) |
| 2 | `"mosca"` | `"canonico"` | 1 `.ply` |
| 3 | `"grilo"` | `"canonico"` | 1 `.ply` |
| 4 | `"mariposa"` | `"dr"` | 2 × `VARIANTES_POR_PIPELINE` `.ply` (metade PipeA, metade PipeB) |
| 5 | `"mosca"` | `"dr"` | idem |
| 6 | `"grilo"` | `"dr"` | idem |

Depois das 6, rode **uma vez** a célula **Separar as pastas por braço**.

O painel só escolhe a pasta e o arquivo de entrada, o número de amostras e se as linhas "Original" do v4 ficam ativas (etapa canônica) ou não (etapa DR). O código dos pipelines é o do v4, sem alteração.

In [ ]:
# [VERSÃO LOCAL] Célula do Colab desativada: ela instalava pacotes e matava o kernel (os.kill),
# o que no PC derrubaria o notebook. Instale uma vez no terminal: pip install numpy scipy plyfile
# # 1. Instalação com blindagem dupla (Numpy e Scipy antigos)
# !pip install  numpy scipy plyfile

# # 2. Comando impiedoso para matar a RAM e forçar o restart do Colab
# import os
# import signal
# print("\n[ATENÇÃO] Matando o kernel para limpar a memória RAM. O Colab vai avisar que a sessão travou/reiniciou. ISSO É ESPERADO E CORRETO.")
# os.kill(os.getpid(), signal.SIGKILL)

In [ ]:
# [VERSÃO LOCAL] Google Drive não é usado no PC.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ============================== PAINEL ==============================
ESPECIME = "mariposa"   # "mariposa", "mosca" ou "grilo"
ETAPA = "canonico"      # "canonico" (1 amostra, sem DR) ou "dr" (lote com PipeA e PipeB)

# Variantes de DR por pipeline e por espécime. A etapa "dr" gera o dobro (o v4 alterna A e B).
VARIANTES_POR_PIPELINE = 500

# Pasta com os modelos canônicos e parte do nome do .ply de cada espécime
# (basta um trecho do nome; espaços e "_" são tratados como iguais)
PASTA_MODELOS = r"C:\Users\felip\Documents\Mestrado\Insetos\TCC modelos 3dgs insetos"
ARQUIVOS = {
    "mariposa": "mari2_hires_SuperLimpa_artigo_final",
    "mosca": "fly",
    "grilo": "grilo limpo",
}
# ====================================================================
import os
assert ESPECIME in ARQUIVOS, f"ESPECIME deve ser um de {list(ARQUIVOS)}"
assert ETAPA in ("canonico", "dr"), "ETAPA deve ser 'canonico' ou 'dr'"

def _norm(t):
    return t.lower().replace(" ", "_")

_todos = sorted(f for f in os.listdir(PASTA_MODELOS) if f.lower().endswith(".ply"))
_candidatos = [f for f in _todos if _norm(ARQUIVOS[ESPECIME]) in _norm(f)]
_exato = [f for f in _candidatos if _norm(os.path.splitext(f)[0]) == _norm(ARQUIVOS[ESPECIME])]
if len(_exato) == 1:
    _candidatos = _exato
if len(_candidatos) != 1:
    raise FileNotFoundError(
        f"Esperava 1 .ply com '{ARQUIVOS[ESPECIME]}' no nome, achei {len(_candidatos)}: {_candidatos}\n"
        f"Arquivos .ply na pasta: {_todos}\nAjuste ARQUIVOS no painel.")
ARQUIVO_ENTRADA = _candidatos[0]
print(f"Execução: {ESPECIME} / {ETAPA} -> {ARQUIVO_ENTRADA}"
      + (f" ({2 * VARIANTES_POR_PIPELINE} amostras)" if ETAPA == "dr" else " (1 amostra)"))

In [ ]:
# ==============================================================================
# UNIFIED DOMAIN RANDOMIZATION FRAMEWORK FOR 3DGS (EXTREME VARIANCE)
# Autors: Prof. João Paulo Gois and Felipe N. C. de Carvalho
# ==============================================================================

# 1. Instalação e Imports

import numpy as np
import os
from plyfile import PlyData, PlyElement
from scipy.spatial.transform import Rotation as R
from scipy.special import expit, logit  # Essencial para conversão matemática do alfa

path_base = PASTA_MODELOS  # [PAINEL]
#path_base = "/content/drive/MyDrive/Mestrado/Gausian Splatting/Artigo versão final /PLY usados no artigo final"
arquivo_entrada = ARQUIVO_ENTRADA  # [PAINEL] antes: "mari2_hires_SuperLimpa_artigo_final.ply"
#arquivo_entrada = "mosquito_limpo_transparencia.ply"

#path_base = "/content/drive/MyDrive/__university/3dgs_samples"
#arquivo_entrada = "barata.ply"

#mosquito_limpo.ply
#barata.ply
#bichopau.ply
#mari1_efa_hires_limpa.ply
caminho_ply_original = os.path.join(path_base, arquivo_entrada)




pasta_saida = os.path.join(path_base, "dataset_sintetico_experimentos")
os.makedirs(pasta_saida, exist_ok=True)

print("\n--- STATUS DO AMBIENTE ---")
if os.path.exists(caminho_ply_original):
    print(f"[OK] Modelo canônico encontrado em: {caminho_ply_original}")
else:
    print(f"[ERRO] Arquivo não encontrado: {caminho_ply_original}")

# ==============================================================================
# MÓDULO 1: SPATIO-OPTICAL PRUNING
# ==============================================================================
def apply_spatio_optical_pruning(vertex_data, alpha_thresh=0.10, anisotropy_thresh=10.0):
    print("--- EXECUTANDO SPATIO-OPTICAL PRUNING ---")

    # 1. Recupera opacidade usando expit (sigmoide do scipy)
    opacidades = expit(vertex_data['opacity'])

    # 2. Recupera escalas reais
    scales = np.vstack([
        np.exp(vertex_data['scale_0']),
        np.exp(vertex_data['scale_1']),
        np.exp(vertex_data['scale_2'])
    ]).T

    s_max = np.max(scales, axis=1)
    s_min = np.min(scales, axis=1)
    s_min[s_min < 1e-6] = 1e-6 # Prevenção div/0

    # 3. Filtro Dual-Criteria
    anisotropia = s_max / s_min
    mascara_floaters = (opacidades < alpha_thresh) & (anisotropia > anisotropy_thresh)

    canonical_data = vertex_data[~mascara_floaters]
    print(f"Floaters removidos (anomalias): {np.sum(mascara_floaters)}")
    print(f"Splats canônicos válidos: {len(canonical_data)}\n")
    return canonical_data

# ==============================================================================
# MÓDULO 2: PIPELINE A - EXTREME PHOTOMETRIC DR
# ==============================================================================
def get_aggressive_photometric_shift():
    """Gera perturbações DC severas simulando falhas de WB e luzes artificiais"""
    base_shift = np.random.uniform(-1.2, 1.2, 3)

    if np.random.rand() > 0.8: # 20% de chance de contaminação extrema (UV, lasers, flash)
        extreme_cast = np.zeros(3)
        extreme_cast[np.random.randint(0, 3)] = np.random.uniform(1.5, 3.0)
        base_shift += extreme_cast
    return base_shift

def generate_photometric_dr(canonical_data, lambda_scale=1.0, delta_temp=[0,0,0], rot_angles=[0,0,0], specular_scale=1.0):
    new_data = canonical_data.copy()

    # 1. DC Thermal Perturbation Agressiva
    new_data['f_dc_0'] = (canonical_data['f_dc_0'] * lambda_scale) + delta_temp[0]
    new_data['f_dc_1'] = (canonical_data['f_dc_1'] * lambda_scale) + delta_temp[1]
    new_data['f_dc_2'] = (canonical_data['f_dc_2'] * lambda_scale) + delta_temp[2]

    # 2. AC Wigner Rotation (Apenas l=1 para tratabilidade)
    rot_matrix = R.from_euler('xyz', rot_angles, degrees=True).as_matrix()

    for color_idx in range(3):
        offset = color_idx * 15
        idx_y, idx_z, idx_x = f'f_rest_{offset}', f'f_rest_{offset+1}', f'f_rest_{offset+2}'

        vec_x = canonical_data[idx_x]
        vec_y = canonical_data[idx_y]
        vec_z = canonical_data[idx_z]

        sh1_vectors = np.vstack((vec_x, vec_y, vec_z)).T
        rotated_sh1 = sh1_vectors.dot(rot_matrix.T)

        new_data[idx_x] = rotated_sh1[:, 0] * specular_scale
        new_data[idx_y] = rotated_sh1[:, 1] * specular_scale
        new_data[idx_z] = rotated_sh1[:, 2] * specular_scale

        # 3. Escala para Alta Ordem (l > 1)
        for m in range(3, 15):
            new_data[f'f_rest_{offset+m}'] = canonical_data[f'f_rest_{offset+m}'] * specular_scale

    return new_data

# ==============================================================================
# MÓDULO 3: PIPELINE B - PROCEDURAL & VOLUMETRIC DR
# ==============================================================================

def generate_procedural_dr(canonical_data, omega_seed, c_amp=3.0, c_shift=[-1.5, -1.5, -1.5]):
    new_data = canonical_data.copy()
    mu_x, mu_y, mu_z = canonical_data['x'], canonical_data['y'], canonical_data['z']

    # --- VARIÂNCIA EXTREMA DE ARQUITETURA (Sorteada por modelo) ---
    # 1. Randomiza o número de oitavas: de 1 (formas ultra lisas) a 5 (formas super ruidosas)
    num_octaves = np.random.randint(1, 6)

    # 2. Randomiza a persistência (decaimento da amplitude):
    # Valores baixos (0.15) geram formas gelatinosas e suaves.
    # Valores altos (0.75) geram ruído caótico/estilhaçado.
    persistence = np.random.uniform(0.15, 0.75)

    # 3. Randomiza a lacunaridade (taxa de crescimento da frequência):
    # Controla o quanto o detalhe encolhe a cada camada.
    lacunarity = np.random.uniform(1.5, 3.5)

    def spatial_noise(x, y, z, seed):
        np.random.seed(seed)
        noise = np.zeros_like(x)

        # 4. Faixas dinâmicas de estiramento (de listras longas a bolhas compactas)
        stretch_x = np.random.uniform(0.2, 4.0)
        stretch_y = np.random.uniform(0.2, 4.0)
        stretch_z = np.random.uniform(0.2, 4.0)

        amplitude = 1.0
        # Frequência inicial variando agressivamente entre macro (1.0) e micro (20.0)
        current_frequency = np.random.uniform(1.0, 20.0)

        for _ in range(num_octaves):
            nx = np.sin((x * stretch_x) * current_frequency + np.random.uniform(0, 2*np.pi))
            ny = np.cos((y * stretch_y) * current_frequency + np.random.uniform(0, 2*np.pi))
            nz = np.sin((z * stretch_z) * current_frequency + np.random.uniform(0, 2*np.pi))

            noise += (nx * ny * nz) * amplitude

            amplitude *= persistence
            current_frequency *= lacunarity

        return noise

    # Sementes geram formas espaciais desalinhadas para os canais
    noise_r = spatial_noise(mu_x, mu_y, mu_z, omega_seed)
    noise_g = spatial_noise(mu_x, mu_y, mu_z, omega_seed + 1000)
    noise_b = spatial_noise(mu_x, mu_y, mu_z, omega_seed + 2000)

    # Normalização dinâmica baseada no número de oitavas para evitar clipar tudo no teto
    max_possible_noise = sum([persistence**i for i in range(num_octaves)])
    noise_r = np.clip(noise_r / (max_possible_noise + 1e-5), -1.0, 1.0)
    noise_g = np.clip(noise_g / (max_possible_noise + 1e-5), -1.0, 1.0)
    noise_b = np.clip(noise_b / (max_possible_noise + 1e-5), -1.0, 1.0)

    # 5. Mistura de Técnicas de Corte (Heaviside vs Contínuo)
    # 50% de chance de manchas com bordas cartoon/vivas (> 0)
    # 50% de chance de manchas com degradês suaves e misturas de cores
    if np.random.rand() > 0.5:
        noise_r = (noise_r > np.random.uniform(-0.4, 0.4)).astype(float)
        noise_g = (noise_g > np.random.uniform(-0.4, 0.4)).astype(float)
        noise_b = (noise_b > np.random.uniform(-0.4, 0.4)).astype(float)

    choice_r = np.random.choice([-1, 1])
    choice_g = np.random.choice([-1, 1])
    choice_b = np.random.choice([-1, 1])

    # Multiplicador de contraste agressivo (c_amp dinâmico por rodada)
    dynamic_amp = c_amp * np.random.uniform(0.3, 1.5)

    new_data['f_dc_0'] = noise_r * (choice_r * dynamic_amp) + c_shift[0]
    new_data['f_dc_1'] = noise_g * (choice_g * dynamic_amp) + c_shift[1]
    new_data['f_dc_2'] = noise_b * (choice_b * dynamic_amp) + c_shift[2]

    # Zera componentes AC (Mantém o visual Matte)
    for color_idx in range(3):
        for m in range(15):
            new_data[f'f_rest_{color_idx*15+m}'] = 0.0

    return new_data


# ==============================================================================
# EXECUÇÃO PRINCIPAL
# ==============================================================================
if os.path.exists(caminho_ply_original):
    ply_data = PlyData.read(caminho_ply_original)
    raw_vertex_data = ply_data.elements[0].data
    canonical_data = apply_spatio_optical_pruning(raw_vertex_data)

    TOTAL_AMOSTRAS = 1 if ETAPA == "canonico" else 2 * VARIANTES_POR_PIPELINE  # [PAINEL]
    print(f"--- INICIANDO GERAÇÃO: {TOTAL_AMOSTRAS} ARQUIVOS ---")

    for i in range(TOTAL_AMOSTRAS):
        # Divisão equitativa (50/50) entre Fotometria e Morfologia
        tipo = i % 2

        if tipo == 0:
            # Pipeline A: Extremo Fotométrico
            scale_light = np.random.uniform(0.2, 3.0)
            tint_light = get_aggressive_photometric_shift()
            rot_light = [np.random.uniform(0, 360), np.random.uniform(0, 360), np.random.uniform(0, 360)]
            spec_scale = np.random.lognormal(mean=0.8, sigma=0.5)

            final_data = generate_photometric_dr(
                canonical_data, lambda_scale=scale_light,
                delta_temp=tint_light, rot_angles=rot_light, specular_scale=spec_scale
            )
            nome_final = "PipeA_Photometric"

        else:
            # Pipeline B: Alta Frequência Procedural
            omega_seed = i * 10
            c_amp = np.random.uniform(2.0, 5.0)
            c_shift = np.random.uniform(-2.0, 2.0, 3)

            final_data = generate_procedural_dr(
                canonical_data, omega_seed=omega_seed, c_amp=c_amp, c_shift=c_shift
            )
            nome_final = "PipeB_Procedural"

        if ETAPA == "canonico":  # [PAINEL] mesmas linhas de antes, ativas só na etapa canônica
            final_data = canonical_data
            nome_final = "Original"

        # Exportação direta
        out_path = os.path.join(pasta_saida, f"sintetico_{nome_final}_{i:03d}.ply")
        el = PlyElement.describe(final_data, 'vertex')
        PlyData([el], text=False).write(out_path)

        if (i + 1) % 50 == 0 or i == 0:
            print(f"Progresso: {i + 1}/{TOTAL_AMOSTRAS} modelos salvos...")

    print(f"\n[SUCESSO] Lote massivo concluído em: {pasta_saida}")

## Renomear saída (depois de cada execução)

In [ ]:
# ======================== RENOMEAR SAÍDA ========================
# O v4 sempre grava "sintetico_...". Troca o prefixo pelo espécime, para a próxima
# execução não sobrescrever estes arquivos e para a Unity separar as classes.
from pathlib import Path

saida = Path(pasta_saida)
gerados = sorted(saida.glob("sintetico_*.ply"))
if not gerados:
    print("Nada para renomear (nenhum sintetico_*.ply na pasta de saída).")
for f in gerados:
    novo = f.with_name(f.name.replace("sintetico", ESPECIME, 1))
    if novo.exists():
        raise FileExistsError(f"{novo.name} já existe. Esta execução ({ESPECIME}/{ETAPA}) já tinha sido feita?")
    f.rename(novo)
print(f"{len(gerados)} arquivo(s) renomeado(s) para {ESPECIME}_...")
for padrao in ["Original", "PipeA", "PipeB"]:
    print(f"  {ESPECIME}_{padrao}: {len(list(saida.glob(f'{ESPECIME}_{padrao}_*.ply')))}")

## Separar as pastas por braço (uma vez, no final)

In [ ]:
# ================== SEPARAR AS PASTAS POR BRAÇO ==================
# Rodar UMA VEZ, depois das 4 execuções. Cria "TCC modelos 3dgs insetos/bracos" com:
#   C0 = canônicos (usado por C0, C1 e C0+L)
#   A  = todos os PipeA       B = todos os PipeB
#   AB = metade dos PipeA (índices 0, 4, 8...) + metade dos PipeB (índices 1, 5, 9...) por espécime
#        (usado por AB e AB+L). São hard links: aparecem nas duas pastas, mas ocupam espaço uma vez só.
# A e B ficam como arquivo permanente dos lotes (o v4 não tem seed fixa: não apague).
import os
import shutil
from pathlib import Path

saida = Path(pasta_saida)
bracos = Path(path_base) / "bracos"
for b in ["C0", "A", "B", "AB"]:
    (bracos / b).mkdir(parents=True, exist_ok=True)

for f in saida.glob("*_Original_*.ply"):
    shutil.move(str(f), bracos / "C0" / f.name)
for f in saida.glob("*_PipeA_*.ply"):
    shutil.move(str(f), bracos / "A" / f.name)
for f in saida.glob("*_PipeB_*.ply"):
    shutil.move(str(f), bracos / "B" / f.name)

for f in list((bracos / "A").glob("*.ply")) + list((bracos / "B").glob("*.ply")):
    indice = int(f.stem.split("_")[-1])
    if indice % 4 in (0, 1):
        destino = bracos / "AB" / f.name
        if not destino.exists():
            try:
                os.link(f, destino)          # hard link (mesmo disco): não ocupa espaço extra
            except OSError:
                shutil.copy2(f, destino)     # outro disco: copia

n_esp = len(ARQUIVOS)
esperado = {"C0": n_esp, "A": n_esp * VARIANTES_POR_PIPELINE,
            "B": n_esp * VARIANTES_POR_PIPELINE, "AB": n_esp * VARIANTES_POR_PIPELINE}
for b, n in esperado.items():
    qtd = len(list((bracos / b).glob("*.ply")))
    print(f"{b}: {qtd} arquivos {'[OK]' if qtd == n else f'[ATENÇÃO: esperado {n}]'}")
print("Pastas em:", bracos)